In [1]:
!pip install streamlit numpy pandas scipy plotly

In [3]:
pip install dash

Note: you may need to restart the kernel to use updated packages.


In [5]:
import numpy as np
import pandas as pd
import scipy.stats as si
import plotly.express as px
import dash
from dash import dcc, html
from dash.dependencies import Input, Output

# Black-Scholes formula
def black_scholes(S, K, T, r, sigma, option_type='call'):
    d1 = (np.log(S / K) + (r + 0.5 * sigma ** 2) * T) / (sigma * np.sqrt(T))
    d2 = d1 - sigma * np.sqrt(T)
    if option_type == 'call':
        price = (S * si.norm.cdf(d1) - K * np.exp(-r * T) * si.norm.cdf(d2))
    elif option_type == 'put':
        price = (K * np.exp(-r * T) * si.norm.cdf(-d2) - S * si.norm.cdf(-d1))
    return price

def calculate_option_prices(stock_prices, volatilities, K, T, r):
    S_grid, sigma_grid = np.meshgrid(stock_prices, volatilities)
    S_grid = S_grid.flatten()
    sigma_grid = sigma_grid.flatten()

    call_prices = np.array([black_scholes(S, K, T, r, sigma, 'call') for S, sigma in zip(S_grid, sigma_grid)])
    put_prices = np.array([black_scholes(S, K, T, r, sigma, 'put') for S, sigma in zip(S_grid, sigma_grid)])

    call_prices = call_prices.reshape(len(volatilities), len(stock_prices))
    put_prices = put_prices.reshape(len(volatilities), len(stock_prices))

    return call_prices, put_prices

def create_heatmap(df, title, color_label):
    fig = px.imshow(df,
                    labels=dict(x="Stock Price", y="Volatility", color=color_label),
                    color_continuous_scale="RdYlGn")
    fig.update_layout(
        title=title,
        width=800,
        height=800,
        margin=dict(l=40, r=40, b=40, t=40),
        xaxis_title='Stock Price',
        yaxis_title='Volatility'
    )
    fig.update_yaxes(scaleanchor="x", scaleratio=1)
    fig.update_traces(colorbar=dict(title=color_label))
    return fig

# Dash application
app = dash.Dash(__name__)

app.layout = html.Div([
    html.H1("Option Pricing Heatmaps"),
    
    html.Div([
        html.Label("Minimum Stock Price:"),
        dcc.Slider(
            id='min-stock-price-slider',
            min=0,
            max=1000,
            step=1,
            value=50,
            marks={i: str(i) for i in range(0, 1001, 50)}
        ),
        html.Label("Maximum Stock Price:"),
        dcc.Slider(
            id='max-stock-price-slider',
            min=0,
            max=1000,
            step=1,
            value=150,
            marks={i: str(i) for i in range(0, 1001, 50)}
        ),
        html.Label("Minimum Volatility:"),
        dcc.Slider(
            id='min-volatility-slider',
            min=0,
            max=1,
            step=0.01,
            value=0.1,
            marks={i / 100: f"{i / 100:.2f}" for i in range(0, 101, 10)}
        ),
        html.Label("Maximum Volatility:"),
        dcc.Slider(
            id='max-volatility-slider',
            min=0,
            max=1,
            step=0.01,
            value=0.3,
            marks={i / 100: f"{i / 100:.2f}" for i in range(0, 101, 10)}
        ),
        html.Label("Time to Maturity (Years):"),
        dcc.Slider(
            id='maturity-slider',
            min=0.01,
            max=5,
            step=0.01,
            value=1,
            marks={i: f"{i:.2f}" for i in range(1, 6)}
        ),
        html.Label("Risk-Free Rate (Decimal):"),
        dcc.Slider(
            id='rate-slider',
            min=0,
            max=1,
            step=0.01,
            value=0.06,
            marks={i / 100: f"{i / 100:.2f}" for i in range(0, 101, 10)}
        ),
    ], style={'padding': '10px'}),
    
    dcc.Graph(id='call-heatmap'),
    dcc.Graph(id='put-heatmap')
])

@app.callback(
    Output('call-heatmap', 'figure'),
    Output('put-heatmap', 'figure'),
    Input('min-stock-price-slider', 'value'),
    Input('max-stock-price-slider', 'value'),
    Input('min-volatility-slider', 'value'),
    Input('max-volatility-slider', 'value'),
    Input('maturity-slider', 'value'),
    Input('rate-slider', 'value')
)
def update_heatmaps(min_stock, max_stock, min_vol, max_vol, maturity, rate):
    if min_stock >= max_stock or min_vol >= max_vol:
        return {}, {}

    stock_prices = np.linspace(min_stock, max_stock, 10)
    volatilities = np.linspace(min_vol, max_vol, 10)
    K = 100

    call_prices, put_prices = calculate_option_prices(stock_prices, volatilities, K, maturity, rate)

    call_df = pd.DataFrame(call_prices, index=[f'{v:.2f}' for v in volatilities], columns=[f'{s:.2f}' for s in stock_prices])
    put_df = pd.DataFrame(put_prices, index=[f'{v:.2f}' for v in volatilities], columns=[f'{s:.2f}' for s in stock_prices])

    fig_call = create_heatmap(call_df, "Call Option Price Heatmap", "Call Price")
    fig_put = create_heatmap(put_df, "Put Option Price Heatmap", "Put Price")

    return fig_call, fig_put

if __name__ == '__main__':
    app.run_server(debug=True)

In [7]:
app.run_server(port=8051)